In [9]:
import sqlite3

In [10]:
conn = sqlite3.connect('../../01_data_pipeline/security_logs.db')
cursor = conn.cursor()

In [11]:
conn.close()
conn = sqlite3.connect('../../01_data_pipeline/security_logs.db')
cursor = conn.cursor()

In [ ]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS unified_events (
    event_id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp DATETIME,
    process TEXT,
    pid INTEGER,
    thread TEXT,
    type TEXT,
    subsystem TEXT,
    activity TEXT,
    raw_message TEXT,
    message TEXT
)
''')

conn.commit()

In [14]:
import pandas as pd

df = pd.read_csv('../datasets/processed/unified_logs_processed.csv')
for _, row in df.iterrows():
    cursor.execute('''
    INSERT INTO unified_events (timestamp, process, pid, thread, type, subsystem, activity, raw_message, message)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (row['timestamp'], row['process'], row['pid'], row['thread'], row['type'], row['subsystem'], row['activity'], row['raw_message'], row['message']))

conn.commit()

In [16]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp'])

In [17]:
# determine hour of day, day of week, and whether it's a weekend
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = df['day_of_week'] >= 5

In [ ]:
# macOS aware off-hours detection: 8pm to 6am on weekdays, all day weekends
df['off_hours'] = df.apply(
    lambda x: 1 if (x['hour'] < 6 or x['hour'] > 20 or x['is_weekend']) else 0,
    axis=1
)